In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from chiCa.chiCa.chipmunk_analysis_tools import load_trialdata

new_rc_params = {"text.usetex": False, "svg.fonttype": "none"}
mpl.rcParams.update(new_rc_params)
plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["font.size"] = 12

%matplotlib widget
%load_ext autoreload
%autoreload 2

In [2]:
chicadata = load_trialdata(
    "/Users/gabriel/data/GRB006/20240821_121447/chipmunk/GRB006_20240821_121447_chipmunk_DemonstratorAudiTask.mat"
)
trial_ts = pd.read_pickle(
    "/Users/gabriel/data/GRB006/20240821_121447/pre_processed/trial_ts.pkl"
)

chicadata = chicadata[chicadata.response_side.isin([0, 1])]
trial_ts = trial_ts[trial_ts.response_side.isin([0, 1])]

In [6]:
corrected_data = {}
trial_start_offset = chicadata.trial_start_time - trial_ts.trial_start


chicadata["corrected_fixation"] = chicadata.apply(
    lambda row: row.trial_start_time + row.DemonInitFixation, axis=1
)
chicadata["corrected_stim_ts"] = chicadata.apply(
    lambda row: row.trial_start_time + row.stimulus_event_timestamps, axis=1
)
chicadata["corrected_withdrawal"] = chicadata.apply(
    lambda row: row.trial_start_time + row.DemonWaitForWithdrawalFromCenter, axis=1
)
chicadata["corrected_response"] = chicadata.apply(
    lambda row: row.trial_start_time + row.DemonWaitForResponse, axis=1
)

corrected_data["center_entry"] = (
    chicadata.corrected_fixation.apply(lambda x: x[0]) - trial_start_offset
)
corrected_data["stim_ts"] = chicadata.corrected_stim_ts - trial_start_offset
corrected_data["center_exit"] = (
    chicadata.corrected_withdrawal.apply(lambda x: x[-1]) - trial_start_offset
)
corrected_data["response"] = (
    chicadata.corrected_response.apply(lambda x: x[-1]) - trial_start_offset
)


corrected_data = pd.concat(
    (
        pd.DataFrame(corrected_data),
        chicadata[
            [
                "outcome_record",
                "response_side",
                "correct_side",
                "actual_wait_time",
                "preStimDelay",
            ]
        ],
    ),
    axis=1,
)
corrected_data.to_pickle("/Users/gabriel/Downloads/20240821_121447_bpod_ts.pkl")

In [18]:
a = np.hstack(corrected_data.stim_ts.apply(lambda x: x[0]))[:12]
b = [
    89.55855733,
    95.46998,
    100.70418533,
    106.82348,
    123.65108242,
    131.92225209,
    135.77785333,
    138.63627685,
    149.01649022,
    172.864784,
    178.1148776,
    181.26665015,
]
np.mean(b - a)

0.013616887777761377

chicadata["bpod_ts"] = chicadata.apply(
    lambda row: row.trial_start_time + row.stimulus_event_timestamps, axis=1
)
chicadata["first_bpod_ts"] = chicadata.apply(
    lambda row: row.trial_start_time + row.stimulus_event_timestamps[0], axis=1
)
chicadata = chicadata[chicadata.response_side.isin([0, 1])]

In [ ]:
from djchurchland.chipmunk.task import Chipmunk

bpod_data = pd.DataFrame(
    (
        Chipmunk.Trial()
        & 'subject_name = "GRB006"'
        & 'session_datetime LIKE "2024-08-21%"'
    ).fetch("trial_start", "stimulus_event_timestamps", "with_choice", as_dict=True)
)
bpod_data["bpod_ts"] = bpod_data.apply(
    lambda row: row.trial_start + row.stimulus_event_timestamps, axis=1
)
bpod_data = bpod_data[bpod_data.with_choice == 1]
bpod_ts = np.hstack(bpod_data.bpod_ts)

In [ ]:
d = pd.DataFrame(
    (
        Chipmunk.Trial()
        & 'subject_name = "GRB006"'
        & 'session_datetime LIKE "2024-08-21%"'
    ).fetch("trial_start", "stimulus_event_timestamps", "with_choice", as_dict=True)
)
sess_start = d.iloc[0].trial_start
d["corrected_start"] = d.apply(lambda row: row.trial_start - sess_start, axis=1)
d["corrected_bpod_ts"] = d.apply(
    lambda row: row.corrected_start + row.stimulus_event_timestamps, axis=1
)
d = d[d.with_choice == 1]
d_ts = np.hstack(d.corrected_bpod_ts)

In [ ]:
trial_ts = pd.read_pickle(
    "/Users/gabriel/data/GRB006/20240821_121447/pre_processed/trial_ts.pkl"
)
trial_ts = trial_ts[trial_ts.response_side.isin([0, 1])]
nidaq_ts = np.hstack(trial_ts.stim_ts)
first_nidaq_ts = np.hstack(trial_ts.first_stim_ts)

In [ ]:
plt.figure()
plt.hist(
    (chicadata.PreStimPeriod.apply(lambda x: np.diff(x)[0]))
    - (trial_ts.first_stim_ts - trial_ts.center_port_entries.apply(lambda x: x[0])),
    bins=20,
)
plt.xlabel("bpod - nidaq calculated pre stim time (s)")
plt.ylabel("count")
plt.title("negative means nidaq is lagging behind bpod")

In [ ]:
chicadata.PreStimPeriod.apply(lambda x: np.diff(x)[0])

In [ ]:
plt.figure()
plt.hist(
    chicadata.PreStimPeriod.apply(lambda x: np.diff(x)[0]),
    bins=50,
    label="bpod",
    alpha=0.8,
)
plt.hist(
    trial_ts.first_stim_ts - trial_ts.center_port_entries.apply(lambda x: x[0]),
    bins=50,
    label="nidaq",
    alpha=0.8,
)
plt.xlabel("pre stimulus times")
plt.ylabel("count")
plt.legend();

In [ ]:
center_onset = chicadata.DemonWaitForCenterFixation.apply(lambda x: x[-1])
center_stim = chicadata.stimulus_event_timestamps.apply(lambda x: x[0])
center_stim - center_onset

In [ ]:
corrected_bpod_ts = np.hstack(
    bpod_data.bpod_ts - (bpod_data.trial_start - trial_ts.trial_start)
)
corrected_chica_ts = np.hstack(
    chicadata.bpod_ts - (chicadata.trial_start_time - trial_ts.trial_start)
)
corrected_chica_first_stim = np.hstack(
    chicadata.first_bpod_ts - (chicadata.trial_start_time - trial_ts.trial_start)
)
np.diff(corrected_chica_ts).min()

In [ ]:
plt.figure()
# plt.vlines(corrected_bpod_ts + 0.015, 0, 1, label = 'bpod', color='b')
plt.vlines(nidaq_ts, 1, 2, label="nidaq", color="r")
plt.vlines(corrected_chica_ts, 2, 3, label="chica_ts", color="k")
# plt.vlines(raw_nidaq, 0, 1, label="raw nidaq", color="b")
plt.legend()
plt.xlim([89, 99])

In [ ]:
plt.figure()
plt.vlines(first_nidaq_ts, 1, 2, label="nidaq", color="r")
plt.vlines(corrected_chica_first_stim, 2, 3, label="chica_ts", color="k")
plt.legend()
plt.xlim([89, 92])

In [ ]:
first_offsets = []
for nidaq, chica in zip(first_nidaq_ts, corrected_chica_first_stim):
    first_offsets.append(nidaq - chica)

first_offsets = np.array(first_offsets)

In [ ]:
print(f" range of offsets is: {(first_offsets.max() - first_offsets.min()) * 1000:.2f}")

In [ ]:
plt.figure()
plt.hist(first_offsets, bins=50)
plt.title("distribution of first stim offsets")
plt.xlabel("first nidaq - first bpod (s)")
plt.ylabel("count")

In [ ]:
prestims = {
    "nidaq_prestims": trial_ts.stim_ts.apply(lambda x: x[0])
    - trial_ts.center_port_entries.apply(lambda x: x[0]),
    "bpod_prestims": center_stim - center_onset,
}
d = pd.DataFrame(prestims)
d["diff"] = d["nidaq_prestims"] - d["bpod_prestims"]
d

TODO:
- ~~check delay between all the first stims between bpod and nidaq~~
- is there any event happening before the first stim in the bpod timestamps?
- is the alignment between neural and behavioral data okay? look into the load_sync_data function